In [3]:
#observation and aproach data analysis - data support

In [ ]:
from __future__ import annotations

import csv
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# =========================
# CONFIG (edit if needed)
# =========================
BASE_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_2_Performance"
)

METRICS_FILE = BASE_DIR / "run_metrics_v16_stage3_enhanced.csv"
STEPS_FILE = BASE_DIR / "run_steps_v16_stage3_breakdown.csv"

# Raw labels in the dataset -> paper labels
STYLE_MAP = {
    "Emu_Community": "Community",
    "Third-Party": "Third Party",
    "Emu_Custom": "Custom",
    "Emu_Community,GMD": "Community+GMD",
}

# Obs 3.3 discusses these explicitly
OBS33_STYLES = ["Emu_Community", "Third-Party", "Emu_Custom"]

# Caps to match paper behavior / remove artifacts
CUSTOM_RUNTIME_CAP_SECONDS = 24 * 3600
CUSTOM_QUEUE_CAP_SECONDS = 24 * 3600


# =========================
# Helpers
# =========================
def fmt_time(seconds: float | int | None) -> str:
    """Format seconds like the paper (m if <1h else h)."""
    if seconds is None or (isinstance(seconds, float) and np.isnan(seconds)):
        return "–"
    s = float(seconds)
    if s < 3600:
        return f"{s / 60:.1f}m"
    return f"{s / 3600:.1f}h"


def safe_int(x: Any) -> int:
    try:
        return int(x)
    except Exception:
        return 0


def load_metrics(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing metrics file: {path}")
    df = pd.read_csv(path)

    needed = {"styles", "event", "run_conclusion", "run_duration_seconds", "queue_seconds"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Metrics file is missing columns: {sorted(missing)}")

    return df


def load_steps(path: Path) -> pd.DataFrame:
    """
    This file can have a header with fewer columns than the actual rows.
    We parse with csv.reader and enforce the observed 14-column schema.
    """
    if not path.exists():
        raise FileNotFoundError(f"Missing steps file: {path}")

    rows: list[list[str]] = []
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        _header = next(reader, None)
        for r in reader:
            if len(r) == 14:
                rows.append(r)

    cols14 = [
        "full_name",
        "run_id",
        "workflow_identifier",
        "workflow_path",
        "head_sha",
        "styles",
        "job_id",
        "job_name",
        "step_name",
        "bucket",
        "started_at",
        "completed_at",
        "duration_seconds",
        "extracted_at_utc",
    ]
    df = pd.DataFrame(rows, columns=cols14)

    df["run_id"] = pd.to_numeric(df["run_id"], errors="coerce")
    df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
    df = df.dropna(subset=["run_id", "duration_seconds"])

    return df


def style_label(raw_style: str) -> str:
    return STYLE_MAP.get(raw_style, raw_style)


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


# =========================
# Core computations
# =========================
def compute_style_run_counts(metrics: pd.DataFrame) -> pd.DataFrame:
    vc = metrics["styles"].value_counts(dropna=False).rename_axis("raw_style").reset_index(name="runs")
    vc["Style"] = vc["raw_style"].map(style_label)
    # keep both raw and label
    return vc[["raw_style", "Style", "runs"]]


def compute_runtime_availability(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for raw_style in STYLE_MAP.keys():
        sub = metrics[metrics["styles"] == raw_style]
        runs = len(sub)

        dur_raw = sub["run_duration_seconds"].dropna()
        n_non_null = len(dur_raw)

        dur_used = dur_raw
        if raw_style == "Emu_Custom":
            dur_used = dur_used[dur_used <= CUSTOM_RUNTIME_CAP_SECONDS]
        n_used = len(dur_used)

        rows.append(
            {
                "raw_style": raw_style,
                "Style": style_label(raw_style),
                "Runs": safe_int(runs),
                "N_runtime_non_null": safe_int(n_non_null),
                "N_runtime_used": safe_int(n_used),
            }
        )
    return pd.DataFrame(rows).set_index("Style")


def compute_queue_availability(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for raw_style in STYLE_MAP.keys():
        sub = metrics[metrics["styles"] == raw_style]
        runs = len(sub)

        q_raw = sub["queue_seconds"]
        n_total = len(q_raw)  # denom for Queue>0
        q = q_raw.fillna(0)

        n_gt0 = int((q > 0).sum())
        nz_raw = q[q > 0]
        n_nz_raw = len(nz_raw)

        nz_used = nz_raw
        if raw_style == "Emu_Custom":
            nz_used = nz_used[nz_used <= CUSTOM_QUEUE_CAP_SECONDS]
        n_nz_used = len(nz_used)

        rows.append(
            {
                "raw_style": raw_style,
                "Style": style_label(raw_style),
                "Runs": safe_int(runs),
                "N_queue_total": safe_int(n_total),
                "N_queue_gt0": safe_int(n_gt0),
                "N_queue_nz_raw": safe_int(n_nz_raw),
                "N_queue_nz_used": safe_int(n_nz_used),
            }
        )
    return pd.DataFrame(rows).set_index("Style")


def obs33_trigger_distribution(metrics: pd.DataFrame) -> pd.DataFrame:
    out_rows = []
    for raw_style in OBS33_STYLES:
        sub = metrics[metrics["styles"] == raw_style]
        n_runs_style = len(sub)
        if n_runs_style == 0:
            continue
        vc = sub["event"].value_counts(dropna=False)

        for event_name, count in vc.items():
            out_rows.append(
                {
                    "raw_style": raw_style,
                    "Style": style_label(raw_style),
                    "TotalRunsStyle": safe_int(n_runs_style),
                    "event": str(event_name),
                    "runs": safe_int(count),
                    "pct": float(count) / n_runs_style * 100.0,
                }
            )
    df = pd.DataFrame(out_rows)
    return df.sort_values(["Style", "pct"], ascending=[True, False])


def table7_availability(metrics: pd.DataFrame) -> pd.DataFrame:
    cats = ["success", "failure", "cancelled", "startup_failure"]
    rows = []
    for raw_style, label in STYLE_MAP.items():
        sub = metrics[metrics["styles"] == raw_style]
        n = len(sub)
        vc = sub["run_conclusion"].value_counts(dropna=False)

        row = {"raw_style": raw_style, "Style": label, "Runs": safe_int(n)}
        cat_counts = {c: safe_int(vc.get(c, 0)) for c in cats}

        for c in cats:
            row[f"{c}_n"] = cat_counts[c]
            row[f"{c}_pct"] = (cat_counts[c] / n * 100.0) if n else np.nan

        row["other_or_nan_n"] = safe_int(n - sum(cat_counts.values()))
        row["other_or_nan_pct"] = ((n - sum(cat_counts.values())) / n * 100.0) if n else np.nan
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Style")
    return df.loc[["Community", "Third Party", "Custom", "Community+GMD"]]


def success_excluding_startup(metrics: pd.DataFrame, raw_style: str) -> dict[str, Any]:
    sub = metrics[metrics["styles"] == raw_style]
    n_total = len(sub)
    if n_total == 0:
        return {"raw_style": raw_style, "Style": style_label(raw_style), "pct": np.nan, "n_success": 0, "denom": 0,
                "n_startup": 0, "n_total": 0}

    n_startup = int((sub["run_conclusion"] == "startup_failure").sum())
    n_success = int((sub["run_conclusion"] == "success").sum())
    denom = int(n_total - n_startup)
    pct = (n_success / denom * 100.0) if denom > 0 else np.nan

    return {
        "raw_style": raw_style,
        "Style": style_label(raw_style),
        "pct": pct,
        "n_success": n_success,
        "denom": denom,
        "n_startup": n_startup,
        "n_total": int(n_total),
    }


def table8_runtime_and_queue(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for raw_style, label in STYLE_MAP.items():
        sub = metrics[metrics["styles"] == raw_style]
        n_runs = len(sub)

        # runtime
        dur_raw = sub["run_duration_seconds"].dropna()
        n_runtime_non_null = len(dur_raw)
        dur = dur_raw
        if raw_style == "Emu_Custom":
            dur = dur[dur <= CUSTOM_RUNTIME_CAP_SECONDS]
        n_runtime_used = len(dur)

        median = np.median(dur) if n_runtime_used else np.nan
        p95 = np.percentile(dur, 95) if n_runtime_used else np.nan
        p99 = np.percentile(dur, 99) if n_runtime_used else np.nan

        # queue
        q_raw = sub["queue_seconds"]
        n_queue_total = len(q_raw)
        q = q_raw.fillna(0)

        n_queue_gt0 = int((q > 0).sum())
        queue_gt0_pct = (n_queue_gt0 / n_queue_total * 100.0) if n_queue_total else np.nan

        nz_raw = q[q > 0]
        n_queue_nz_raw = len(nz_raw)
        nz = nz_raw
        if raw_style == "Emu_Custom":
            nz = nz[nz <= CUSTOM_QUEUE_CAP_SECONDS]
        n_queue_nz_used = len(nz)
        qmed = np.median(nz) if n_queue_nz_used else np.nan

        rows.append(
            {
                "raw_style": raw_style,
                "Style": label,
                "Runs": safe_int(n_runs),
                "N_runtime_non_null": safe_int(n_runtime_non_null),
                "N_runtime_used": safe_int(n_runtime_used),
                "N_queue_total": safe_int(n_queue_total),
                "N_queue_gt0": safe_int(n_queue_gt0),
                "N_queue_nz_raw": safe_int(n_queue_nz_raw),
                "N_queue_nz_used": safe_int(n_queue_nz_used),
                "Median": fmt_time(median),
                "P95": fmt_time(p95),
                "P99": fmt_time(p99),
                "Queue>0_pct": queue_gt0_pct,
                "Queue>0": f"{queue_gt0_pct:.2f}%" if not (isinstance(queue_gt0_pct, float) and np.isnan(queue_gt0_pct)) else "–",
                "Qmed(nz)": fmt_time(qmed) if not (isinstance(qmed, float) and np.isnan(qmed)) else "–",
            }
        )

    df = pd.DataFrame(rows).set_index("Style")
    return df.loc[["Community", "Third Party", "Custom", "Community+GMD"]]


def obs35_pct_over_one_hour(metrics: pd.DataFrame, raw_style: str) -> dict[str, Any]:
    sub = metrics[metrics["styles"] == raw_style]
    dur = sub["run_duration_seconds"].dropna()
    n_runtime = len(dur)
    n_over_1h = int((dur > 3600).sum())
    pct = (n_over_1h / n_runtime * 100.0) if n_runtime else np.nan
    return {
        "raw_style": raw_style,
        "Style": style_label(raw_style),
        "pct": pct,
        "N_runtime": safe_int(n_runtime),
        "N_over_1h": safe_int(n_over_1h),
    }


def obs36_step_bucket_shares(steps: pd.DataFrame) -> pd.DataFrame:
    buckets = ["third_party", "env_setup", "artifact", "test", "other"]
    rows = []
    for raw_style, label in STYLE_MAP.items():
        sub = steps[steps["styles"] == raw_style]
        if sub.empty:
            rows.append(
                {
                    "raw_style": raw_style,
                    "Style": label,
                    "StepRows": 0,
                    "RunsWithSteps": 0,
                    "total_extracted_seconds": 0.0,
                    **{b: np.nan for b in buckets},
                }
            )
            continue

        n_step_rows = len(sub)
        n_runs_with_steps = int(sub["run_id"].nunique())
        total = float(sub["duration_seconds"].sum())
        sums = sub.groupby("bucket")["duration_seconds"].sum()

        row = {
            "raw_style": raw_style,
            "Style": label,
            "StepRows": safe_int(n_step_rows),
            "RunsWithSteps": safe_int(n_runs_with_steps),
            "total_extracted_seconds": total,
        }
        for b in buckets:
            row[b] = (float(sums.get(b, 0.0)) / total * 100.0) if total > 0 else np.nan
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Style")
    return df.loc[["Community", "Third Party", "Custom", "Community+GMD"]]


def compute_overall_step_coverage(steps: pd.DataFrame) -> dict[str, int]:
    return {
        "total_step_rows": int(len(steps)),
        "runs_with_steps_total": int(steps["run_id"].nunique()) if not steps.empty else 0,
    }


# =========================
# Report assembly + export
# =========================
@dataclass
class RQ3Report:
    # General population stats (approach)
    total_runs_metrics: int
    total_step_rows: int
    runs_with_steps_total: int

    # Per-style run counts (population)
    runs_by_style: dict[str, int]

    # Metric availability
    runtime_non_null_by_style: dict[str, int]
    runtime_used_by_style: dict[str, int]
    steps_runs_with_steps_by_style: dict[str, int]
    steps_rows_by_style: dict[str, int]

    # Observation/Result tables (structured)
    obs33_triggers: list[dict[str, Any]]
    table7_availability: list[dict[str, Any]]
    table8_runtime_queue: list[dict[str, Any]]
    obs35_over_1h: list[dict[str, Any]]
    obs36_step_bucket_shares: list[dict[str, Any]]


def build_report(metrics: pd.DataFrame, steps: pd.DataFrame) -> RQ3Report:
    total_runs_metrics = int(len(metrics))
    overall_step_cov = compute_overall_step_coverage(steps)

    # run counts by style
    run_counts_df = compute_style_run_counts(metrics)
    runs_by_style = {row["Style"]: int(row["runs"]) for _, row in run_counts_df.iterrows()}

    # runtime availability
    runtime_av = compute_runtime_availability(metrics)
    runtime_non_null_by_style = runtime_av["N_runtime_non_null"].to_dict()
    runtime_used_by_style = runtime_av["N_runtime_used"].to_dict()

    # step coverage per style + bucket shares
    step_shares = obs36_step_bucket_shares(steps)
    steps_runs_with_steps_by_style = step_shares["RunsWithSteps"].to_dict()
    steps_rows_by_style = step_shares["StepRows"].to_dict()

    # Obs / tables
    obs33 = obs33_trigger_distribution(metrics)

    t7 = table7_availability(metrics)
    t8 = table8_runtime_and_queue(metrics)

    obs35 = [
        obs35_pct_over_one_hour(metrics, "Emu_Community"),
        obs35_pct_over_one_hour(metrics, "Third-Party"),
    ]

    # Convert dataframes to list-of-dicts for JSON
    obs33_records = obs33.to_dict(orient="records")
    t7_records = t7.reset_index().to_dict(orient="records")
    t8_records = t8.reset_index().to_dict(orient="records")
    obs36_records = step_shares.reset_index().to_dict(orient="records")

    return RQ3Report(
        total_runs_metrics=total_runs_metrics,
        total_step_rows=int(overall_step_cov["total_step_rows"]),
        runs_with_steps_total=int(overall_step_cov["runs_with_steps_total"]),
        runs_by_style=runs_by_style,
        runtime_non_null_by_style={k: int(v) for k, v in runtime_non_null_by_style.items()},
        runtime_used_by_style={k: int(v) for k, v in runtime_used_by_style.items()},
        steps_runs_with_steps_by_style={k: int(v) for k, v in steps_runs_with_steps_by_style.items()},
        steps_rows_by_style={k: int(v) for k, v in steps_rows_by_style.items()},
        obs33_triggers=obs33_records,
        table7_availability=t7_records,
        table8_runtime_queue=t8_records,
        obs35_over_1h=obs35,
        obs36_step_bucket_shares=obs36_records,
    )


def render_text_report(rep: RQ3Report) -> str:
    # Pretty, paper-facing text summary for copy/paste
    lines: list[str] = []
    lines.append("=== RQ3 Population + Observations 3.3–3.6 (Auto-Extracted Report) ===\n")
    lines.append("---- General population (Approach) ----")
    lines.append(f"Loaded metrics rows (runs): {rep.total_runs_metrics:,}")
    lines.append(f"Loaded step rows: {rep.total_step_rows:,}")
    lines.append(f"Runs with step traces: {rep.runs_with_steps_total:,}\n")

    lines.append("Runs by style:")
    for k in ["Community", "Third Party", "Custom", "Community+GMD"]:
        lines.append(f"  - {k}: {rep.runs_by_style.get(k, 0):,}")
    lines.append("")

    lines.append("Runtime availability (non-null duration) by style:")
    for k in ["Community", "Third Party", "Custom", "Community+GMD"]:
        lines.append(f"  - {k}: non-null={rep.runtime_non_null_by_style.get(k, 0):,}, used={rep.runtime_used_by_style.get(k, 0):,}")
    lines.append("")

    lines.append("Step-trace coverage by style:")
    for k in ["Community", "Third Party", "Custom", "Community+GMD"]:
        lines.append(
            f"  - {k}: RunsWithSteps={rep.steps_runs_with_steps_by_style.get(k, 0):,}, StepRows={rep.steps_rows_by_style.get(k, 0):,}"
        )
    lines.append("")

    # Obs 3.3 (top 3 per style)
    lines.append("---- Obs 3.3: Trigger distribution (top 3 per style) ----")
    df33 = pd.DataFrame(rep.obs33_triggers)
    for style in df33["Style"].unique():
        sub = df33[df33["Style"] == style].sort_values("pct", ascending=False).head(3)
        total = int(sub["TotalRunsStyle"].iloc[0]) if not sub.empty else 0
        lines.append(f"{style} (TotalRunsStyle={total:,})")
        for _, r in sub.iterrows():
            lines.append(f"  {r['event']}: {int(r['runs']):,}/{total:,} ({r['pct']:.2f}%)")
        lines.append("")

    # Table 7
    lines.append("---- Table 7: Availability (counts + %) ----")
    df7 = pd.DataFrame(rep.table7_availability)
    # Keep order
    order = ["Community", "Third Party", "Custom", "Community+GMD"]
    df7["__ord"] = df7["Style"].apply(lambda x: order.index(x) if x in order else 999)
    df7 = df7.sort_values("__ord").drop(columns="__ord")

    for _, r in df7.iterrows():
        lines.append(
            f"{r['Style']} (Runs={int(r['Runs']):,}): "
            f"success={int(r['success_n']):,} ({r['success_pct']:.2f}%), "
            f"failure={int(r['failure_n']):,} ({r['failure_pct']:.2f}%), "
            f"cancelled={int(r['cancelled_n']):,} ({r['cancelled_pct']:.2f}%), "
            f"startup_failure={int(r['startup_failure_n']):,} ({r['startup_failure_pct']:.2f}%)"
        )
    lines.append("")

    # Obs 3.4 extra (excluding startup)
    lines.append("Obs 3.4 extra: success rate excluding startup failures")
    for raw_style in ["Third-Party", "Emu_Community"]:
        d = success_excluding_startup_df_row(rep, raw_style)
        lines.append(
            f"{d['Style']}: {d['pct']:.2f}% (success={d['n_success']:,} / denom={d['denom']:,}; "
            f"startup_failures={d['n_startup']:,}; total={d['n_total']:,})"
        )
    lines.append("")

    # Table 8
    lines.append("---- Table 8: Runtime & queueing pressure (with denominators) ----")
    df8 = pd.DataFrame(rep.table8_runtime_queue)
    df8["__ord"] = df8["Style"].apply(lambda x: order.index(x) if x in order else 999)
    df8 = df8.sort_values("__ord").drop(columns="__ord")
    for _, r in df8.iterrows():
        lines.append(
            f"{r['Style']} (Runs={int(r['Runs']):,}): "
            f"Median={r['Median']} (n_runtime={int(r['N_runtime_used']):,}), "
            f"P95={r['P95']}, P99={r['P99']}; "
            f"Queue>0={r['Queue>0']} (n={int(r['N_queue_gt0']):,}/{int(r['N_queue_total']):,}); "
            f"Qmed(nz)={r['Qmed(nz)']} (n_nz={int(r['N_queue_nz_used']):,})"
        )
    lines.append("")

    # Obs 3.5
    lines.append("---- Obs 3.5: % runs exceeding 1 hour (with denominators) ----")
    for r in rep.obs35_over_1h:
        lines.append(f"{r['Style']}: {r['pct']:.2f}% ({r['N_over_1h']:,}/{r['N_runtime']:,} non-null runtimes)")
    lines.append("")

    # Obs 3.6
    lines.append("---- Obs 3.6: Step-time bucket shares (with step/run counts) ----")
    df36 = pd.DataFrame(rep.obs36_step_bucket_shares)
    df36["__ord"] = df36["Style"].apply(lambda x: order.index(x) if x in order else 999)
    df36 = df36.sort_values("__ord").drop(columns="__ord")
    for _, r in df36.iterrows():
        def pct(x: Any) -> str:
            return "–" if pd.isna(x) else f"{float(x):.2f}%"
        lines.append(
            f"{r['Style']}: RunsWithSteps={int(r['RunsWithSteps']):,}, StepRows={int(r['StepRows']):,} | "
            f"third_party={pct(r['third_party'])}, env_setup={pct(r['env_setup'])}, "
            f"artifact={pct(r['artifact'])}, test={pct(r['test'])}, other={pct(r['other'])}"
        )

    lines.append("\n=== End of report ===")
    return "\n".join(lines)


def success_excluding_startup_df_row(rep: RQ3Report, raw_style: str) -> dict[str, Any]:
    # recompute from stored table7 counts for consistency
    # raw_style here is dataset raw label (e.g., "Third-Party", "Emu_Community")
    # we need metrics access for exact denom; easiest is to compute from table7 rows:
    # - total n_total is from runs_by_style
    # - n_startup is from table7 availability row
    # - n_success is from table7 availability row
    label = style_label(raw_style)
    n_total = rep.runs_by_style.get(label, 0)
    t7 = pd.DataFrame(rep.table7_availability).set_index("Style")
    if label not in t7.index:
        return {"Style": label, "pct": np.nan, "n_success": 0, "denom": 0, "n_startup": 0, "n_total": n_total}
    n_startup = int(t7.loc[label, "startup_failure_n"])
    n_success = int(t7.loc[label, "success_n"])
    denom = int(n_total - n_startup)
    pct = (n_success / denom * 100.0) if denom > 0 else np.nan
    return {"Style": label, "pct": pct, "n_success": n_success, "denom": denom, "n_startup": n_startup, "n_total": n_total}


def save_outputs(base_dir: Path, rep: RQ3Report) -> dict[str, Path]:
    ensure_dir(base_dir)

    out_paths: dict[str, Path] = {}

    # 1) JSON (structured)
    json_path = base_dir / "rq3_population_and_obs_stats.json"
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(asdict(rep), f, indent=2)
    out_paths["json"] = json_path

    # 2) TXT report (human-readable)
    txt_path = base_dir / "rq3_population_and_obs_stats.txt"
    txt = render_text_report(rep)
    with txt_path.open("w", encoding="utf-8") as f:
        f.write(txt)
    out_paths["txt"] = txt_path

    # 3) CSV exports (tables / obs)
    # Obs 3.3 full distribution
    obs33_path = base_dir / "obs33_trigger_distribution_full.csv"
    pd.DataFrame(rep.obs33_triggers).to_csv(obs33_path, index=False)
    out_paths["obs33_csv"] = obs33_path

    # Table 7
    t7_path = base_dir / "table7_availability_counts_pct.csv"
    pd.DataFrame(rep.table7_availability).to_csv(t7_path, index=False)
    out_paths["table7_csv"] = t7_path

    # Table 8
    t8_path = base_dir / "table8_runtime_queue_denominators.csv"
    pd.DataFrame(rep.table8_runtime_queue).to_csv(t8_path, index=False)
    out_paths["table8_csv"] = t8_path

    # Obs 3.5
    obs35_path = base_dir / "obs35_over_one_hour.csv"
    pd.DataFrame(rep.obs35_over_1h).to_csv(obs35_path, index=False)
    out_paths["obs35_csv"] = obs35_path

    # Obs 3.6
    obs36_path = base_dir / "obs36_step_bucket_shares.csv"
    pd.DataFrame(rep.obs36_step_bucket_shares).to_csv(obs36_path, index=False)
    out_paths["obs36_csv"] = obs36_path

    return out_paths


def main() -> None:
    metrics = load_metrics(METRICS_FILE)
    steps = load_steps(STEPS_FILE)

    rep = build_report(metrics, steps)

    # Print to console
    print(render_text_report(rep))

    # Save outputs in the same root directory as the inputs (BASE_DIR)
    out_paths = save_outputs(BASE_DIR, rep)
    print("\nSaved outputs:")
    for k, p in out_paths.items():
        print(f"  - {k}: {p}")


if __name__ == "__main__":
    main()


=== RQ3 Population + Observations 3.3–3.6 (Auto-Extracted Report) ===

---- General population (Approach) ----
Loaded metrics rows (runs): 31,121
Loaded step rows: 463,565
Runs with step traces: 8,558

Runs by style:
  - Community: 26,054
  - Third Party: 3,100
  - Custom: 363
  - Community+GMD: 51

Runtime availability (non-null duration) by style:
  - Community: non-null=25,630, used=25,630
  - Third Party: non-null=2,099, used=2,099
  - Custom: non-null=363, used=315
  - Community+GMD: non-null=51, used=51

Step-trace coverage by style:
  - Community: RunsWithSteps=7,421, StepRows=363,982
  - Third Party: RunsWithSteps=711, StepRows=91,221
  - Custom: RunsWithSteps=79, StepRows=2,276
  - Community+GMD: RunsWithSteps=15, StepRows=1,305

---- Obs 3.3: Trigger distribution (top 3 per style) ----
Community (TotalRunsStyle=26,054)
  push: 20,662/26,054 (79.30%)
  schedule: 3,928/26,054 (15.08%)
  pull_request: 736/26,054 (2.82%)

Custom (TotalRunsStyle=363)
  push: 297/363 (81.82%)
  pul